# 💬 Week 6: Analisis Sentimen (Sentiment Analysis)

## Tujuan Pembelajaran:
1. Memahami konsep **Analisis Sentimen** — menentukan apakah suatu teks bersifat positif, negatif, atau netral
2. Menerapkan **TextBlob** untuk mengukur polaritas teks secara otomatis
3. Mengkategorikan ulasan berdasarkan sentimen dan membandingkannya dengan **skor bintang** (`score`)
4. Menganalisis **tren jumlah ulasan** dan **tren sentimen** dari waktu ke waktu berdasarkan kolom `at`
5. Membuat **Word Cloud** dari keseluruhan teks ulasan

---
> 📂 **Dataset**: Honest Review (`df_honest_reviews_id.csv`) — kolom `content` (teks ulasan), `score` (bintang 1–5), `at` (tanggal ulasan)
> File ini berada di folder `Week 2/` dalam workspace yang sama.

## 1) Install Library

Install library yang dibutuhkan:
- `textblob` — library NLP ringan untuk analisis sentimen
- `wordcloud` — untuk membuat visualisasi Word Cloud

In [ ]:
!pip install textblob wordcloud pandas matplotlib

## 2) Import Library

Import modul yang diperlukan untuk analisis sentimen dan visualisasi.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from wordcloud import WordCloud

print("Library berhasil diimport!")

## 3) Load Dataset

Load dataset Honest Review dari `df_honest_reviews_id.csv`. Kolom yang digunakan:
- `content` — teks ulasan pengguna
- `score` — rating bintang (1–5)
- `at` — tanggal ulasan dikirim

In [ ]:
file_path = '../Week 2/df_honest_reviews_id.csv'
df = pd.read_csv(file_path, parse_dates=['at'])

print(f"Jumlah data: {len(df)}")
print(f"Kolom: {list(df.columns)}")
df[['content', 'score', 'at']].head()

## 4) Fungsi Analisis Sentimen

**TextBlob** menganalisis sentimen teks berbasis leksikon Inggris. Nilai polaritas:
- **> 0** = Positif
- **= 0** = Netral  
- **< 0** = Negatif

> ⚠️ TextBlob bekerja optimal pada teks Inggris. Untuk teks Bahasa Indonesia, kita juga akan membandingkan hasilnya dengan `score` asli (bintang) sebagai ground truth.

In [ ]:
def analyze_sentiment_textblob(text):
    """Hitung polaritas sentimen menggunakan TextBlob."""
    return TextBlob(str(text)).sentiment.polarity

def categorize_sentiment(polarity):
    """Kategorikan nilai polaritas menjadi label sentimen."""
    if polarity > 0:
        return 'Positif'
    elif polarity < 0:
        return 'Negatif'
    else:
        return 'Netral'

def score_to_sentiment(score):
    """Konversi skor bintang (1-5) ke label sentimen."""
    if score >= 4:
        return 'Positif'
    elif score == 3:
        return 'Netral'
    else:
        return 'Negatif'

print("Fungsi sentimen siap!")

## 5) Terapkan Sentimen ke Seluruh Dataset

Terapkan fungsi ke kolom `content` dan tambahkan kolom `polarity`, `sentiment_textblob`, serta `sentiment_score`.

In [ ]:
# Sentimen dari TextBlob
df['polarity']           = df['content'].apply(analyze_sentiment_textblob)
df['sentiment_textblob'] = df['polarity'].apply(categorize_sentiment)

# Sentimen dari score bintang (ground truth)
df['sentiment_score']    = df['score'].apply(score_to_sentiment)

print("Kolom baru berhasil ditambahkan!")
df[['content', 'score', 'polarity', 'sentiment_textblob', 'sentiment_score']].head(5)

## 6) Distribusi Polaritas

Histogram distribusi nilai polaritas TextBlob di seluruh ulasan. Apakah ulasan cenderung positif?

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['polarity'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Netral (0)')
plt.xlabel('Polaritas', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)
plt.title('Distribusi Polaritas Sentimen Honest Review (TextBlob)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Rata-rata polaritas : {df['polarity'].mean():.4f}")
print(f"Polaritas maks      : {df['polarity'].max():.4f}")
print(f"Polaritas min       : {df['polarity'].min():.4f}")

## 7) Distribusi Kategori Sentimen

Bandingkan distribusi sentimen dari TextBlob versus dari skor bintang asli.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Positif': '#4CAF50', 'Netral': '#2196F3', 'Negatif': '#F44336'}

for ax, col, title in [
    (axes[0], 'sentiment_textblob', 'Sentimen TextBlob'),
    (axes[1], 'sentiment_score',    'Sentimen dari Skor Bintang'),
]:
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[colors.get(k, '#999') for k in counts.index], edgecolor='white')
    ax.bar_label(bars, fmt='%d', fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Kategori Sentimen')
    ax.set_ylabel('Jumlah Ulasan')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribusi Sentimen Honest Review', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8) Tren Jumlah Ulasan dari Waktu ke Waktu

Grafik jumlah ulasan per bulan — memperlihatkan periode mana yang paling banyak ulasan.

In [ ]:
df_time = df.set_index('at').resample('ME')['content'].count().reset_index()
df_time.columns = ['bulan', 'jumlah_ulasan']

plt.figure(figsize=(14, 5))
plt.plot(df_time['bulan'], df_time['jumlah_ulasan'], 'o-', color='steelblue', linewidth=2, markersize=5)
plt.fill_between(df_time['bulan'], df_time['jumlah_ulasan'], alpha=0.15, color='steelblue')
plt.xlabel('Bulan', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)
plt.title('Tren Jumlah Ulasan Honest Review per Bulan', fontsize=14)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9) Tren Sentimen dari Waktu ke Waktu

Grafik rata-rata polaritas per bulan dan persentase sentimen positif/negatif.

In [ ]:
df_trend = df.set_index('at').resample('ME')['polarity'].mean().reset_index()
df_trend.columns = ['bulan', 'avg_polarity']

plt.figure(figsize=(14, 5))
pos = df_trend['avg_polarity'].clip(lower=0)
neg = df_trend['avg_polarity'].clip(upper=0)
plt.fill_between(df_trend['bulan'], pos, alpha=0.4, color='green', label='Polaritas Positif')
plt.fill_between(df_trend['bulan'], neg, alpha=0.4, color='red',   label='Polaritas Negatif')
plt.plot(df_trend['bulan'], df_trend['avg_polarity'], 'k-', linewidth=1.5, label='Rata-rata Polaritas')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.xlabel('Bulan', fontsize=12)
plt.ylabel('Rata-rata Polaritas', fontsize=12)
plt.title('Tren Sentimen Honest Review per Bulan', fontsize=14)
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10) Word Cloud

Visualisasikan kata yang paling sering muncul dalam ulasan menggunakan Word Cloud.

In [ ]:
all_text = ' '.join(df['content'].dropna().astype(str))

wc = WordCloud(
    width=900, height=450, max_words=200,
    background_color='white', colormap='RdYlGn',
    collocations=False
).generate(all_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Honest Review', fontsize=16)
plt.tight_layout()
plt.show()

## 11) Ringkasan Statistik

Cetak statistik akhir dari analisis sentimen.

In [ ]:
total = len(df)
print("=== Ringkasan Analisis Sentimen Honest Review ===")
print(f"Total ulasan         : {total:,}")
print(f"Rata-rata polaritas  : {df['polarity'].mean():.4f}")
print()
print("Distribusi Sentimen (TextBlob):")
for cat, cnt in df['sentiment_textblob'].value_counts().items():
    print(f"  {cat:<10} : {cnt:>6,} ({cnt/total*100:.1f}%)")
print()
print("Distribusi Sentimen (Skor Bintang):")
for cat, cnt in df['sentiment_score'].value_counts().items():
    print(f"  {cat:<10} : {cnt:>6,} ({cnt/total*100:.1f}%)")